# Rawat PHIE — generate ALL results (Regimes A & B)

This notebook runs the **same pipeline** as the Anaconda Prompt command, from Jupyter.

**Before you start:** keep these four files in the *same folder as this notebook*:
`run_all_regimes.py`, `rawat_full_pipeline.py`, `regime_a_benchmark.py`, `phie_aqpso_lowo.py`.

**Install once (in the same environment as Jupyter):**
```
pip install numpy pandas scikit-learn matplotlib lasio lightgbm shap
```
LightGBM + SHAP give the paper's model and attribution; without them the code falls back to scikit-learn (runs, but the numbers are not the paper's).

- **Regime B** (blind-well / LOWO) → the paper's primary evidence: Tables 1, 7, 8, 9, 14, 15, 16 and Figs 2, 5–13, 15, 16, 20.
- **Regime A** (within-well optimizer benchmark, *diagnostic*) → Tables 10–13 and Fig 14.

## 1. Edit these paths, then run every cell

In [ ]:
# EDIT THESE FOUR LINES ---------------------------------------------------
LAS_RAW  = r"D:\data\las_raw"                      # raw wireline LAS folder
LAS_CALC = r"D:\data\las_calc"  # calculated LAS folder
OUT      = r"D:\rawat_local\results_all"                                # output folder
MODEL    = "auto"      # 'auto' uses LightGBM if installed, else scikit-learn fallback
STAGES   = "all"       # 'all' = both regimes | 'b' = blind only | 'a' = benchmark only
# ------------------------------------------------------------------------
# Leave these at the paper defaults (do NOT enable fast for real runs):
FAST = False
ROW_STRIDE = 1

## 2. Run both regimes

In [ ]:
from run_all_regimes import run_all

results = run_all(las_raw=LAS_RAW, las_calc=LAS_CALC, out=OUT,
                  stages=STAGES, model=MODEL, fast=FAST, row_stride=ROW_STRIDE)
results

## 3. Check the headline numbers and list the artifacts

Sanity check: the deterministic `PHIT x (1-VCL)` value should read ~**0.936** (it is model-independent, so it must match the manuscript — confirming the LAS merge is correct). Pooled blind LOWO R² should be ~**0.96** when run with LightGBM at full resolution.

In [ ]:
import json, pandas as pd
from pathlib import Path
OUT = Path(OUT)
print(json.dumps(json.load(open(OUT/'SUMMARY_all_regimes.json')), indent=2))
print('\nMANIFEST (every output -> paper label):')
display(pd.read_csv(OUT/'MANIFEST.csv'))

## 4. Preview a few figures inline

In [ ]:
from IPython.display import Image, display
for stem in ['Fig_08_lowo_pooled_crossplot', 'Fig_07_baseline_ladder',
             'Fig_13_leakage_gauge', 'Fig_14_optimizer_convergence']:
    p = OUT/'figures'/f'{stem}.png'
    if p.exists():
        print(stem); display(Image(str(p)))

## 5. Paste-ready tables

`TABLES.md` (Regime B) renders every table in Markdown. Regime A tables are the `Table_10..13_*.csv` files in `tables/`. Replace the `[bracketed]` placeholders in the manuscript with these values.

**Not produced here:** SHAP beeswarm/dependence and LIME panels (Figs 17–19) need `shap`/`lime`; generate them separately. Everything else for both regimes is in your `OUT` folder.